In [1]:
import numpy as np
import scipy.stats as stats
import src.noise as noise
import src.release as dprelease

from src.analysis import NoisyContingencyTable

Here's a contingency table.

In [2]:
true_tbl = np.array([
    [65, 109],
    [243, 1348]
])

It's straightforward to have `scipy` calculate the 95% confidence interval of its odds ratio.

In [3]:
scipy_ci = stats.contingency.odds_ratio(true_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (2.363633, 4.629785)


The `dpvalue` library can do this too. Although it uses a _credible interval_ because it's drawing from a posterior. It implicitly converts `true_tbl` into a table with elements of type `NoisyFloat` (the noise is just zero).

In [5]:
nct = NoisyContingencyTable(true_tbl)
dpvalue_lo, dpvalue_hi = nct.with_sampling_uncertainty().odds_ratio().credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (2.424268, 4.538868)


More or less consistent, I guess.

Let's add some noise and pretend the contingency table is a differentially private release.

In [6]:
noise_factory = noise.gaussian(loc=0.0, scale=12.0)
noisy_tbl = dprelease.noisy_float_array(true_tbl, noise_factory)
noisy_tbl

array([[~59.450114473415276, ~115.15102353018418],
       [~247.50586948830602, ~1352.432559000632]], dtype=object)

What would `scipy` do with this? Since `scipy` isn't DP-noise-aware, it can only take the observed values as truth and account for sampling uncertainty as it did before.

But oops, we can't use `scipy` on this directly, even though `NoisyFloat` is castable to `float`. The elements are counts, and have to be type `int`. Let's extract the noisy observations.

In [7]:
noisy_int_tbl = [
    [int(noisy_tbl[0, 0]._obs), int(noisy_tbl[0, 1]._obs)],
    [int(noisy_tbl[1, 0]._obs), int(noisy_tbl[1, 1]._obs)]]

noisy_int_tbl

[[59, 115], [247, 1352]]

And here's the `scipy` confidence interval again.

In [8]:
scipy_ci = stats.contingency.odds_ratio(noisy_int_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (1.994992, 3.952998)


But the `dpvalue` library can use `noisy_tbl` directly. It accounts for both uncertainty from DP noise _and_ uncertainty from sampling.

In [9]:
nct_noisy = NoisyContingencyTable(noisy_tbl)
dpvalue_lo, dpvalue_hi = nct_noisy.with_sampling_uncertainty().odds_ratio().credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (1.464477, 4.853038)


But it takes almost 8 seconds to compute this...